In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# code flow
* load the dataset
* basic preprocessing
* training process
   * create the model
   * forward pass
   * loss calculate
   * parameters update
* model evaluate

In [3]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [ ]:
df.shape

(569, 31)

In [ ]:
df.columns

Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       'target'],
      dtype='object')

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, :-1], df.iloc[:, -1], test_size=0.2)

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Numpy arrays to pytorch tensors

In [6]:
X_train_tensor = torch.from_numpy(X_train_scaled)
X_test_tensor = torch.from_numpy(X_test_scaled)
y_train_tensor = torch.from_numpy(y_train.values)
y_test_tensor = torch.from_numpy(y_test.values)

In [21]:
from torch.utils.data import DataLoader, Dataset

In [22]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self,index):
    return self.features[index], self.labels[index]


In [23]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [24]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# Define the model

In [33]:
import torch.nn as nn

In [34]:
class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    output = self.linear(features)
    output = self.sigmoid(output)

    return output


In [35]:
lr = 0.1
epochs = 25

# Training Pipeline

In [25]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

# define loss function
loss_function = nn.BCELoss()

In [31]:
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:
     # Convert batch_features and batch_labels to the correct dtype and shape (same as dtype of weights)
     batch_features = batch_features.float()
     batch_labels = batch_labels.float().unsqueeze(1) # Reshape target shape to match prediction shape

     # forward pass
     y_pred = model(batch_features)

     # loss calculate
     loss = loss_function(y_pred, batch_labels)

     # clear gradients
     optimizer.zero_grad()

     # backward pass
     loss.backward()

     # parameters update
     optimizer.step()


  # print loss for each epoch
  print(f'Epoch: {epoch + 1}, loss: {loss.item()}')

Epoch: 1, loss: 0.08456635475158691
Epoch: 2, loss: 0.1016681119799614
Epoch: 3, loss: 0.052575308829545975
Epoch: 4, loss: 0.15352757275104523
Epoch: 5, loss: 0.02127336524426937
Epoch: 6, loss: 0.010685933753848076
Epoch: 7, loss: 0.019789040088653564
Epoch: 8, loss: 0.16480489075183868
Epoch: 9, loss: 0.014987870119512081
Epoch: 10, loss: 0.06588289886713028
Epoch: 11, loss: 0.02335960976779461
Epoch: 12, loss: 0.32321763038635254
Epoch: 13, loss: 0.032287485897541046
Epoch: 14, loss: 0.009920818731188774
Epoch: 15, loss: 0.009452096186578274
Epoch: 16, loss: 0.027943303808569908
Epoch: 17, loss: 0.0342685841023922
Epoch: 18, loss: 0.016303669661283493
Epoch: 19, loss: 0.11913981288671494
Epoch: 20, loss: 0.010482517071068287
Epoch: 21, loss: 0.2522261440753937
Epoch: 22, loss: 0.02944333478808403
Epoch: 23, loss: 0.10287386924028397
Epoch: 24, loss: 0.009893474169075489
Epoch: 25, loss: 0.028636928647756577


# Evaluation

In [40]:

# model evaluation using test_loader
model.eval() # set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
  for batch_features, batch_labels in test_loader:

    batch_features = batch_features.float()
    batch_labels = batch_labels.float()

    # forward pass
    y_pred = model.forward(batch_features)
    y_pred = (y_pred > 0.5).float()

    # calculate accuracy for the current batch
    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

  # calculate overall accuracy
  overall_accuracy = sum(accuracy_list) / len(accuracy_list)
  print(f'Accuracy: {overall_accuracy:.4f}')


0.96875
0.9375
0.96875
0.8888888955116272
Accuracy: 0.9410
